# Denoising-step study (standalone)

Sweeps `runner.sample(..., steps=N, solver=...)` over a range of fixed-step
and adaptive ODE solvers and reports both the probe-based flux RMSE and the
decoded-integral flux RMSE from `runner.evaluate`. Extracted from
`neurips_diff_eval.ipynb` so it can be run on its own without the FID /
warm-restart sections.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os

sys.path.append("..")
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [ ]:
import omegaconf, yaml
from collections import defaultdict

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

from neugk.diffusion import get_diffusion_runner

from neurips_diff_eval import (
    compute_statistics,
    compute_fid,
    compute_fid_on_latents,
    extract_gyroswin_latents,
    to_model_space,
    from_model_space,
    run_trajectory_pair,
    time_to_convergence,
    plot_correlation_grid,
    load_reference_flux,
)

In [ ]:
DATA_PATH = "/local00/bioinf/galletti/preprocessed_kvikio"
AE_CHECKPOINT = "/restricteddata/ukaea/checkpoints/neurips26/AE_noCond/20260405_022851_327/best.pth"
# Same scaling-law GyroSwin checkpoint used by neurips_fid_gyroswin_latents.ipynb
GYROSWIN_CHECKPOINT = "/restricteddata/ukaea/checkpoints/scaling_law/gyroswin_xxl_fluxavg_cond_nodrop_l1"
GKW_RAW_DIR = "/restricteddata/ukaea/gyrokinetics/raw"

# pretrained diffusion model
PRETRAINED_DIR = "/restricteddata/ukaea/checkpoints/neurips26/DIFF_FLOW/20260412_180101_948/"

ID_VAL = [
    "iteration_262.h5",
    "iteration_135.h5",
    "iteration_8.h5",
    "iteration_232.h5",
    "iteration_148.h5",
    "iteration_115.h5",
]
OOD_VAL = [f"ood_iteration_{i}.h5" for i in range(5)]
TRAIN_TRAJS = "iteration_{0-5,7-12,14-31,33-82,84-99}.h5"

N_EPOCHS = 10
BATCH_SIZE = 256
LR = 2e-3
VAL_EVERY = 30
N_DENOISING_STEPS = 10

MINIBATCH_OT = True
NOISE_DISTRIBUTION = "gaussian"  # "gaussian" or "mixture"
CONTINUOUS_TIME = True

FID_MODE = "gyroswin"  # "ae" or "gyroswin"
FID_N_COMPONENTS = 512
FID_MAX_SAMPLES = 128
GEN_BATCH_SIZE = 32


In [ ]:
if PRETRAINED_DIR:
    print(f"loading pretrained from {PRETRAINED_DIR}")
    pretrained_cfg = omegaconf.OmegaConf.load(os.path.join(PRETRAINED_DIR, "config.yaml"))
    pretrained_cfg.output_path = PRETRAINED_DIR
    pretrained_cfg.dataset.path = DATA_PATH
    pretrained_cfg.ae_checkpoint = os.path.dirname(AE_CHECKPOINT)
    pretrained_cfg.dataset.gds_override = True
    pretrained_cfg.dataset.validation_trajectories = ID_VAL
    pretrained_cfg.dataset.val_subsample = 1
    pretrained_cfg.dataset.eval_cond_filters = {}
    pretrained_cfg.validation.probe = {"targets": []}
    pretrained_cfg.logging.writer = None
    pretrained_cfg.logging.tqdm = True
    pretrained_cfg.training.num_workers = 0
    pretrained_cfg.training.pin_memory = False
    pretrained_cfg.ddp.enable = False
    pretrained_cfg.deepspeed.enable = False
    runner = get_diffusion_runner(rank=0, cfg=pretrained_cfg, world_size=1)
    torch.use_deterministic_algorithms(False)
    ckp = torch.load(
        os.path.join(PRETRAINED_DIR, "best.pth"),
        map_location=runner.device,
        weights_only=False,
    )
    runner.model.load_state_dict(ckp["model_state_dict"])
    runner.model.eval()
    print(f"--> loaded diffusion weights from epoch {ckp.get('epoch', '?')}")
else:
    print("training from scratch")
    cfg = omegaconf.OmegaConf.load(os.path.join(os.path.dirname(AE_CHECKPOINT), "config.yaml"))
    with open("dit_config.yaml", "r") as f:
        diff_cfg = omegaconf.DictConfig(yaml.safe_load(f))
    cfg.model = diff_cfg.model
    cfg.ddp = diff_cfg.ddp
    cfg.workflow = "diffusion"
    cfg.output_path = "/tmp/diffusion_eval_pipeline"
    cfg.dataset.path = DATA_PATH
    cfg.dataset.gds_override = False
    cfg.dataset.backend = "gds"
    cfg.ae_checkpoint = os.path.dirname(AE_CHECKPOINT)
    cfg.dataset.training_trajectories = TRAIN_TRAJS
    cfg.dataset.validation_trajectories = ID_VAL
    cfg.dataset.val_subsample = 1
    cfg.dataset.eval_cond_filters = {}
    cfg.model.latent_dim = 512
    cfg.training.batch_size = BATCH_SIZE
    cfg.training.learning_rate = LR
    cfg.training.n_epochs = N_EPOCHS
    cfg.validation.validate_every_n_epochs = VAL_EVERY
    cfg.validation.probe = {"targets": []}
    cfg.logging.writer = None
    cfg.logging.tqdm = True
    cfg.training.num_workers = 0
    cfg.training.pin_memory = False
    cfg.model.diffusion.minibatch_ot = MINIBATCH_OT
    cfg.model.diffusion.noise_distribution = NOISE_DISTRIBUTION
    cfg.model.diffusion.continuous_time = CONTINUOUS_TIME
    cfg.ddp.enable = False
    cfg.deepspeed.enable = False
    runner = get_diffusion_runner(rank=0, cfg=cfg, world_size=1)

print(f"Model: {sum(p.numel() for p in runner.model.parameters())/1e6:.1f}M params")
print(f"Train: {len(runner.trainset)}, Val: {sum(len(v) for v in runner.valsets)}")


In [ ]:
from neurips_diff_eval import (
    collect_latents,
    generate_latents,
    fit_probes,
    plot_probes,
    encode_valset,
    plot_val_probe,
)

PROBE_N_COMPONENTS = 64
PROBE_ALPHA = 1.0
PROBE_SUBSAMPLE = 256

_cond_keys = sorted(runner.cfg.model.conditioning)
X_ae, y_flux, C_train = collect_latents(runner.trainset.precomputed_latents, _cond_keys)
print(f"training set: {X_ae.shape[0]} samples, latent dim={X_ae.shape[1]}")

if PROBE_SUBSAMPLE and PROBE_SUBSAMPLE < len(X_ae):
    idx = np.random.choice(len(X_ae), PROBE_SUBSAMPLE, replace=False)
    X_ae, y_flux, C_train = X_ae[idx], y_flux[idx], C_train[idx]
    print(f"subsampled to {PROBE_SUBSAMPLE} samples")

X_gen = generate_latents(runner, C_train, batch_size=GEN_BATCH_SIZE, steps=N_DENOISING_STEPS)

probes = fit_probes(
    X_ae,
    X_gen,
    y_flux,
    _cond_keys,
    C_train,
    n_components=PROBE_N_COMPONENTS,
    alpha=PROBE_ALPHA,
)
print(
    f"PCA: {X_ae.shape[1]} -> {probes['pca'].n_components_} ({probes['pca'].explained_variance_ratio_.sum():.1%} var)"
)
print(f"flux probe — RMSE ae: {probes['flux']['rmse_ae']:.4f}, RMSE gen: {probes['flux']['rmse_gen']:.4f}")
for k in _cond_keys:
    print(f"  {k} — RMSE ae: {probes['cond']['rmse'][k]['ae']:.4f}, RMSE gen: {probes['cond']['rmse'][k]['gen']:.4f}")

_ = plot_probes(y_flux, probes, _cond_keys, C_train, X_ae, X_gen)

X_val_ae, C_val, y_val_gt, val_fi = encode_valset(
    runner.valsets[0],
    runner.autoencoder,
    _cond_keys,
    runner.device,
    batch_size=GEN_BATCH_SIZE,
)
X_val_gen = generate_latents(runner, C_val, batch_size=GEN_BATCH_SIZE, steps=N_DENOISING_STEPS)

pca = probes["pca"]
pred_val_ae = probes["flux"]["probe_ae"].predict(pca.transform(X_val_ae))
pred_val_gen = probes["flux"]["probe_gen"].predict(pca.transform(X_val_gen))
rmse_val_ae = np.sqrt(((y_val_gt - pred_val_ae) ** 2).mean())
rmse_val_gen = np.sqrt(((y_val_gt - pred_val_gen) ** 2).mean())
print(f"flux probe (val) — RMSE ae: {rmse_val_ae:.4f}, RMSE gen: {rmse_val_gen:.4f}")

_ = plot_val_probe(
    runner.valsets[0],
    pred_val_ae,
    pred_val_gen,
    y_val_gt,
    val_fi,
    rmse_val_ae,
    rmse_val_gen,
)

## 1.6 Denoising-step study

In [ ]:
DENOISING_STEPS_STUDY = [1, 2, 4, 8, 10, 16, 20, 32]
DEFAULT_STEPS = N_DENOISING_STEPS  # the run-config default we want to highlight

ADAPTIVE_SOLVER = "rk4"   # torchdiffeq method
ADAPTIVE_RTOL   = 1e-4
ADAPTIVE_ATOL   = 1e-5

flux_rmse_probe = {}
for n in DENOISING_STEPS_STUDY:
    Xg = generate_latents(runner, C_val, batch_size=GEN_BATCH_SIZE, steps=n)
    pred = probes["flux"]["probe_gen"].predict(probes["pca"].transform(Xg))
    rmse = float(np.sqrt(((y_val_gt - pred) ** 2).mean()))
    flux_rmse_probe[n] = rmse
    print(f"  [probe]  steps={n:3d}  flux RMSE = {rmse:.4f}")

# Adaptive (torchdiffeq) sample: integrate the velocity field with `dopri5`
runner.model.eval()
gen_lats_adapt = []
with torch.no_grad():
    for i in range(0, len(C_val), GEN_BATCH_SIZE):
        c = torch.as_tensor(C_val[i:i + GEN_BATCH_SIZE], dtype=torch.float32, device=runner.device)
        z = runner.sample(c, latent_only=True, steps=2,
                          solver=ADAPTIVE_SOLVER, rtol=ADAPTIVE_RTOL, atol=ADAPTIVE_ATOL)
        gen_lats_adapt.append(z.cpu().numpy().reshape(z.shape[0], -1))
Xg_adapt = np.concatenate(gen_lats_adapt, axis=0)
pred_adapt = probes["flux"]["probe_gen"].predict(probes["pca"].transform(Xg_adapt))
flux_rmse_probe_adapt = float(np.sqrt(((y_val_gt - pred_adapt) ** 2).mean()))
print(f"  [probe]  adaptive ({ADAPTIVE_SOLVER}, rtol={ADAPTIVE_RTOL}, atol={ADAPTIVE_ATOL})  "
      f"flux RMSE = {flux_rmse_probe_adapt:.4f}")

# ----- (2) decoded-integral flux RMSE vs steps via runner.evaluate ---------
import omegaconf as _omegaconf
import functools as _functools

_eval_metrics_by_steps = {}
_orig_eval_steps = getattr(runner.cfg.validation, "eval_sample_steps", None)
runner.model.eval()
try:
    for n in DENOISING_STEPS_STUDY:
        with _omegaconf.open_dict(runner.cfg.validation):
            runner.cfg.validation.eval_sample_steps = n
        log_metrics, _, _ = runner.evaluate(epoch=0, evaluate_probing=False, no_save=True)
        _eval_metrics_by_steps[n] = dict(log_metrics)
        for k in (k for k in log_metrics if "flux" in k.lower() and "rmse" in k.lower()):
            print(f"  [evaluate] steps={n:3d}  {k} = {log_metrics[k]:.4f}")

    # Adaptive evaluate: monkey-patch runner.sample to force the adaptive solver
    _orig_sample = runner.sample
    def _adaptive_sample(condition, steps=None, latent_only=False, **kw):
        return _orig_sample(
            condition, steps=steps or 2, latent_only=latent_only,
            solver=ADAPTIVE_SOLVER, rtol=ADAPTIVE_RTOL, atol=ADAPTIVE_ATOL,
        )
    runner.sample = _adaptive_sample
    try:
        with _omegaconf.open_dict(runner.cfg.validation):
            runner.cfg.validation.eval_sample_steps = 2  # # of output points; solver picks substeps
        log_metrics_adapt, _, _ = runner.evaluate(epoch=0, evaluate_probing=False, no_save=True)
        _eval_metrics_adapt = dict(log_metrics_adapt)
        for k in (k for k in log_metrics_adapt if "flux" in k.lower() and "rmse" in k.lower()):
            print(f"  [evaluate] adaptive ({ADAPTIVE_SOLVER})  {k} = {log_metrics_adapt[k]:.4f}")
    finally:
        runner.sample = _orig_sample
finally:
    with _omegaconf.open_dict(runner.cfg.validation):
        if _orig_eval_steps is None:
            if "eval_sample_steps" in runner.cfg.validation:
                del runner.cfg.validation["eval_sample_steps"]
        else:
            runner.cfg.validation.eval_sample_steps = _orig_eval_steps

# Auto-pick the flux-RMSE keys
all_keys = set()
for m in _eval_metrics_by_steps.values():
    all_keys.update(m.keys())
flux_rmse_keys = sorted(k for k in all_keys
                          if "flux" in k.lower() and "rmse" in k.lower())

# ----- combined plot ------------------------------------------------------
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.4), sharex=False)

x_adapt = float(DENOISING_STEPS_STUDY[-1]) * 2.0

xs = list(flux_rmse_probe.keys())
ys = [flux_rmse_probe[k] for k in xs]
axL.plot(xs, ys, "o-", lw=1.6, color="#264653", markersize=6, label="fixed-step Euler")
for x, y in zip(xs, ys):
    axL.annotate(f"{y:.3f}", (x, y), textcoords="offset points", xytext=(0, 8),
                 ha="center", fontsize=8)
axL.scatter([x_adapt], [flux_rmse_probe_adapt], s=110, marker="*",
            color="#8a3ffc", edgecolor="black", lw=0.6, zorder=6,
            label=f"adaptive ({ADAPTIVE_SOLVER})")
axL.annotate(f"{flux_rmse_probe_adapt:.3f}", (x_adapt, flux_rmse_probe_adapt),
             textcoords="offset points", xytext=(0, 10), ha="center", fontsize=8)
if DEFAULT_STEPS in flux_rmse_probe:
    axL.axvline(DEFAULT_STEPS, color="#e76f51", ls="--", lw=1.2, alpha=0.85,
                label=f"default = {DEFAULT_STEPS}")
    axL.scatter([DEFAULT_STEPS], [flux_rmse_probe[DEFAULT_STEPS]],
                s=130, facecolors="none", edgecolors="#e76f51", lw=2.0, zorder=5)
axL.set_xscale("log", base=2)
axL.set_xticks(xs + [x_adapt])
axL.set_xticklabels([str(x) for x in xs] + [f"{ADAPTIVE_SOLVER}"], rotation=0, fontsize=8)
axL.set_xlabel("# denoising steps")
axL.set_ylabel("validation flux RMSE (probe)")
axL.set_title("Probe-based flux RMSE")
axL.grid(True, which="both", alpha=0.25)
axL.legend(fontsize=8)

if flux_rmse_keys:
    cmap = plt.get_cmap("viridis")
    for i, k in enumerate(flux_rmse_keys):
        ys = [_eval_metrics_by_steps[n].get(k, np.nan) for n in DENOISING_STEPS_STUDY]
        if not any(np.isfinite(ys)):
            continue
        col = cmap(0.15 + 0.7 * i / max(len(flux_rmse_keys) - 1, 1))
        axR.plot(DENOISING_STEPS_STUDY, ys, "o-", lw=1.6, color=col,
                 markersize=6, label=k)
        if DEFAULT_STEPS in DENOISING_STEPS_STUDY:
            axR.scatter([DEFAULT_STEPS],
                        [_eval_metrics_by_steps[DEFAULT_STEPS].get(k, np.nan)],
                        s=130, facecolors="none", edgecolors="#e76f51", lw=2.0, zorder=5)
        # Adaptive point for this metric.
        v_adapt = _eval_metrics_adapt.get(k, np.nan) if "_eval_metrics_adapt" in dir() else np.nan
        if np.isfinite(v_adapt):
            axR.scatter([x_adapt], [v_adapt], s=120, marker="*",
                        color=col, edgecolor="black", lw=0.6, zorder=6)
            axR.annotate(f"{v_adapt:.3f}", (x_adapt, v_adapt),
                         textcoords="offset points", xytext=(0, 10), ha="center",
                         fontsize=8)
    axR.axvline(DEFAULT_STEPS, color="#e76f51", ls="--", lw=1.2, alpha=0.85,
                label=f"default = {DEFAULT_STEPS}")
    axR.legend(fontsize=8, loc="best")
else:
    axR.text(0.5, 0.5, "no flux-RMSE key in runner.evaluate output",
             ha="center", va="center", transform=axR.transAxes, fontsize=10)
axR.set_xscale("log", base=2)
axR.set_xticks(DENOISING_STEPS_STUDY + [x_adapt])
axR.set_xticklabels([str(x) for x in DENOISING_STEPS_STUDY] + [f"{ADAPTIVE_SOLVER}"],
                     rotation=0, fontsize=8)
axR.set_xlabel("# denoising steps")
axR.set_ylabel("validation flux RMSE (decoded integral)")
axR.set_title("runner.evaluate flux RMSE")
axR.grid(True, which="both", alpha=0.25)
fig.tight_layout()


In [ ]:
SOLVER_CONFIGS = [
    # fixed-step references
    {"solver": "rk4",           "steps": 2},
    {"solver": "rk4",           "steps": 4},
    {"solver": "rk4",           "steps": 8},
    # adaptive solvers — `steps` is just the output grid
    {"solver": "adaptive_heun", "steps": 2, "rtol": 1e-4, "atol": 1e-5},
    {"solver": "bosh3",         "steps": 2, "rtol": 1e-4, "atol": 1e-5},
    {"solver": "dopri5",        "steps": 2, "rtol": 1e-3, "atol": 1e-4},
    {"solver": "dopri5",        "steps": 2, "rtol": 1e-5, "atol": 1e-6},
    {"solver": "dopri8",        "steps": 2, "rtol": 1e-5, "atol": 1e-6},
]

import omegaconf as _omegaconf
runner.model.eval()
_orig_sample_method = runner.sample  # bound method, restored at end
try:

    def _probe_rmse(solver, steps, rtol=None, atol=None):
        lats = []
        with torch.no_grad():
            for i in range(0, len(C_val), GEN_BATCH_SIZE):
                c = torch.as_tensor(C_val[i:i + GEN_BATCH_SIZE], dtype=torch.float32, device=runner.device)
                kw = dict(latent_only=True, steps=steps, solver=solver)
                if rtol is not None: kw["rtol"] = rtol
                if atol is not None: kw["atol"] = atol
                z = runner.sample(c, **kw)
                lats.append(z.cpu().numpy().reshape(z.shape[0], -1))
        Xg = np.concatenate(lats, axis=0)
        pred = probes["flux"]["probe_gen"].predict(probes["pca"].transform(Xg))
        return float(np.sqrt(((y_val_gt - pred) ** 2).mean()))

    def _eval_rmse(solver, steps, rtol=None, atol=None):
        orig_eval_steps = getattr(runner.cfg.validation, "eval_sample_steps", None)
        orig_sample = runner.sample
        if solver != "euler":
            def _patched(condition, steps=None, latent_only=False, **kw):
                return orig_sample(condition, steps=steps or 2, latent_only=latent_only,
                                   solver=solver, rtol=rtol, atol=atol)
            runner.sample = _patched
        try:
            with _omegaconf.open_dict(runner.cfg.validation):
                runner.cfg.validation.eval_sample_steps = steps
            log_metrics, _, _ = runner.evaluate(epoch=0, evaluate_probing=False, no_save=True)
        finally:
            runner.sample = orig_sample
            with _omegaconf.open_dict(runner.cfg.validation):
                if orig_eval_steps is None:
                    if "eval_sample_steps" in runner.cfg.validation:
                        del runner.cfg.validation["eval_sample_steps"]
                else:
                    runner.cfg.validation.eval_sample_steps = orig_eval_steps
        return {k: v for k, v in log_metrics.items()
                if "flux" in k.lower() and "rmse" in k.lower()}

    def _label(cfg):
        s = f"{cfg['solver']}/s={cfg['steps']}"
        if "rtol" in cfg: s += f" r={cfg['rtol']:.0e}"
        if "atol" in cfg: s += f" a={cfg['atol']:.0e}"
        return s

    # Run all configs.
    playground_points = []
    for cfg in SOLVER_CONFIGS:
        label = _label(cfg)
        print(f"=== {label} ===")
        try:
            probe_rmse_run = _probe_rmse(**cfg)
            eval_rmse_run  = _eval_rmse(**cfg)
            print(f"  probe RMSE     : {probe_rmse_run:.4f}")
            for k, v in eval_rmse_run.items():
                print(f"  {k:30s}: {v:.4f}")
            playground_points.append({
                "label": label, "cfg": cfg,
                "probe_rmse": probe_rmse_run,
                "eval_rmse":  eval_rmse_run,
            })
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")

    x_adapt = float(DENOISING_STEPS_STUDY[-1]) * 2.0  # placeholder for cell-20 dopri5 star

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 5), sharex=False)

    # Probe panel.
    xs = list(flux_rmse_probe.keys())
    ys = [flux_rmse_probe[k] for k in xs]
    axL.plot(xs, ys, "o-", lw=1.6, color="#264653", markersize=6, label="Euler (cell-20)")
    axL.scatter([x_adapt], [flux_rmse_probe_adapt], s=110, marker="*",
                color="#8a3ffc", edgecolor="black", lw=0.6, zorder=6,
                label=f"cell-20 adaptive ({ADAPTIVE_SOLVER})")
    if DEFAULT_STEPS in flux_rmse_probe:
        axL.axvline(DEFAULT_STEPS, color="#e76f51", ls="--", lw=1.2, alpha=0.85,
                    label=f"default = {DEFAULT_STEPS}")
        axL.scatter([DEFAULT_STEPS], [flux_rmse_probe[DEFAULT_STEPS]],
                    s=130, facecolors="none", edgecolors="#e76f51", lw=2.0, zorder=5)

    # Sweep markers — one shape per solver name, viridis-coloured by config index.
    solver_markers = {"rk4": "s", "euler": "o", "dopri5": "*", "dopri8": "P",
                      "bosh3": "X", "adaptive_heun": "^"}
    sweep_cmap = plt.get_cmap("plasma")
    for i, pt in enumerate(playground_points):
        col = sweep_cmap(0.1 + 0.8 * i / max(len(playground_points) - 1, 1))
        marker = solver_markers.get(pt["cfg"]["solver"], "D")
        axL.scatter([pt["cfg"]["steps"]], [pt["probe_rmse"]],
                    s=140, marker=marker, color=col, edgecolor="black", lw=0.6,
                    zorder=7, label=pt["label"])
    axL.set_xscale("log", base=2)
    axL.set_xlabel("# steps  (output grid points for adaptive)")
    axL.set_ylabel("validation flux RMSE (probe)")
    axL.set_title("Probe-based flux RMSE")
    axL.grid(True, which="both", alpha=0.25)
    axL.legend(fontsize=7, loc="best", ncol=2)

    # Evaluate panel — overlay sweep markers per metric.
    if flux_rmse_keys:
        cmap = plt.get_cmap("viridis")
        for i, k in enumerate(flux_rmse_keys):
            ys_k = [_eval_metrics_by_steps[n].get(k, np.nan) for n in DENOISING_STEPS_STUDY]
            if not any(np.isfinite(ys_k)): continue
            col = cmap(0.15 + 0.7 * i / max(len(flux_rmse_keys) - 1, 1))
            axR.plot(DENOISING_STEPS_STUDY, ys_k, "o-", lw=1.6, color=col,
                     markersize=6, label=f"Euler {k}")
            v_adapt = _eval_metrics_adapt.get(k, np.nan)
            if np.isfinite(v_adapt):
                axR.scatter([x_adapt], [v_adapt], s=110, marker="*",
                            color=col, edgecolor="black", lw=0.6, zorder=6)
            for j, pt in enumerate(playground_points):
                v_pg = pt["eval_rmse"].get(k, np.nan)
                if not np.isfinite(v_pg): continue
                pg_col = sweep_cmap(0.1 + 0.8 * j / max(len(playground_points) - 1, 1))
                marker = solver_markers.get(pt["cfg"]["solver"], "D")
                axR.scatter([pt["cfg"]["steps"]], [v_pg], s=120, marker=marker,
                            color=pg_col, edgecolor="black", lw=0.6, zorder=7,
                            label=(pt["label"] if i == 0 else None))
        axR.axvline(DEFAULT_STEPS, color="#e76f51", ls="--", lw=1.2, alpha=0.85,
                    label=f"default = {DEFAULT_STEPS}")
        axR.legend(fontsize=6, loc="best", ncol=2)
    else:
        axR.text(0.5, 0.5, "no flux-RMSE key in runner.evaluate output",
                 ha="center", va="center", transform=axR.transAxes, fontsize=10)
    axR.set_xscale("log", base=2)
    axR.set_xlabel("# steps  (output grid points for adaptive)")
    axR.set_ylabel("validation flux RMSE (decoded integral)")
    axR.set_title("runner.evaluate flux RMSE")
    axR.grid(True, which="both", alpha=0.25)
    fig.tight_layout()

    print("\n=== probe-RMSE leaderboard (low is better) ===")
    sorted_pts = sorted(playground_points, key=lambda p: p["probe_rmse"])
    for p in sorted_pts:
        print(f"  {p['probe_rmse']:.4f}  {p['label']}")
finally:
    runner.sample = _orig_sample_method  # always restore Euler default
